Brainstorm

Hierarchies
From minimal

THE WAY
Predetermine levels
Take subsets like 3, 6, 9... not every step is necessary
Random sampling like 5-10 times for feasible runtime

Metrics
filtered mean
median / other percentiles
difference filtered mean
difference percentiles
signs count


In [ ]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root

import pandas as pd
import numpy as np
import plotly.express as px
import python.utils as utils
import python.plots as plots
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS, TimeSeriesType
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, MetricsIndex, LoadedErrorMetrics, ErrorDifference, FilteredErrorMetrics, ClonedErrorMetrics, CustomMetrics
from python.error_metrics import MetricsIndex as MI # Alias
from python.forecast_reconciliation import ForecastReconciliation

from python.aggregation import MONTHS, FORTNIGHTS, WEEKS, WEEKDAYS, WEEKENDS, DAYS, TWO_DAYS, FOUR_DAYS, PeriodicAggregation, ArbitraryAggregation, Aggregation
from python.temporal_hierarchy import TemporalHierarchy

In [ ]:
available_levels: list[Aggregation] = [
    TWO_DAYS,
    FOUR_DAYS,
    WEEKDAYS,
    WEEKENDS,
    WEEKS,
    FORTNIGHTS,
    ArbitraryAggregation('8Days1', [1] * 8 + [0] * 20),
    ArbitraryAggregation('8Days2', [0] * 8 + [1] * 8 + [0] * 12),
    ArbitraryAggregation('8Days3', [0] * 16 + [1] * 8 + [0] * 4),
    ArbitraryAggregation('16Days', [1] * 16 + [0] * 12),
    ArbitraryAggregation('12Days', [0] * 16 + [1] * 12),
    PeriodicAggregation('11L', 1, 1, True), # Skip every other
    PeriodicAggregation('11R', 1, 1, False),
    PeriodicAggregation('43L', 4, 3, True), # 4 Days on, 3 skip
    PeriodicAggregation('43R', 4, 3, False),
    PeriodicAggregation('34L', 3, 4, True), # 3 Days on, 4 skip
    PeriodicAggregation('34R', 3, 4, False),
    PeriodicAggregation('61L', 6, 1, True), # 6 Days on, 1 skip
    ArbitraryAggregation('AllWeekends', [0,0,0,0,0,1,1] * 4),
    ArbitraryAggregation('AllWeekdays', [1,1,1,1,1,0,0] * 4),
    ArbitraryAggregation('AllExtendedWeekends', [0,0,0,0,1,1,1] * 4),
    ArbitraryAggregation('AllExtendedWeekdays', [1,1,1,1,1,1,0] * 4),
    ArbitraryAggregation('10', [1, 0] * 14), # Zig zag
    ArbitraryAggregation('01', [0, 1] * 14)
]

In [ ]:
len(available_levels)

21 available levels: samples of: 3, 6, 9, 12, 15, 18, 21
Plus one Minimal and AllPossible

In [ ]:
from python.sampled_hierarchies import SampledHierarchies

sh = SampledHierarchies(available_levels, 3, 10)

In [ ]:
all_possible = TemporalHierarchy([MONTHS] + available_levels + [DAYS],'AllPossible')

In [ ]:
X_raw = utils.fetch_dataset_web_traffic()
X = utils.filter_last_days_available(X_raw, 28*100)
X = utils.reduce_dataset(X, 13000)
X = utils.align_fill_and_trim_series(X)

In [ ]:
h = 28 * 3

all_possible.resample_time_series(X)

In [ ]:
all_possible.make_base_forecasts(h)

In [ ]:
for h in sh.hierarchies:
    h.copy_data(all_possible)

In [ ]:
from datetime import datetime

for h in sh.hierarchies:
    print(f'{datetime.now().isoformat()} - {h.name}')
    ForecastReconciliation(h).reconcile('shr')
    errors = ErrorMetrics(h)
    errors.calc_error_metrics([DAYS, MONTHS])
    errors.save(f'sample/metrics-{h.name}.csv')

In [ ]:
from pathlib import Path
ems = []
for file in Path("sample").rglob("*"):
    if file.is_file():
        ems.append(LoadedErrorMetrics(file))

ems.append(FilteredErrorMetrics(LoadedErrorMetrics('metrics/base_hier_errors.csv'), MI(hierarchy='Minimal')))




In [ ]:
MergedErrorMetrics(ems).save('metrics/sample_hiers_errors_raw.csv')

Cleaning out multiple base forecast errors

In [ ]:
em = MergedErrorMetrics(ems)

em = MergedErrorMetrics([
    FilteredErrorMetrics(em, MI(forecast_type=RECONCILED_FORECASTS)),
    FilteredErrorMetrics(em, MI(forecast_type=BASE_FORECASTS, hierarchy='Minimal'))
])

em.save('metrics/sample_hiers_errors.csv')